In [20]:
import sys
import hashlib
import logging
import pathlib

import numpy as np
import pandas as pd

In [22]:
df = pd.read_csv("sample_deliveries.csv")

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (512, 12)


In [23]:
df.head()

,Delivery ID,Starting Time,Van Model,Num Packages,Town,Start Lat,Start Lon,End Lat,End Lon,Temperature,Conditions,Delivery Time
0,1,2026-01-01 00:00:00,Van A,4,Mumbai,18.608279,73.819797,18.912351,74.340687,34.4,Clear,1701.0
1,2,2026-01-01 00:37:00,Van C,19,Ahilyanagar,18.657992,73.828884,18.561228,74.626233,18.5,Cloudy,2603.0
2,3,2026-01-01 01:14:00,Van B,28,Ahilyanagar,18.593986,73.838392,19.110447,74.154642,21.6,Rain,3592.0
3,4,2026-01-01 01:51:00,Van B,26,Mumbai,18.783891,74.443483,19.049402,74.405023,22.8,Cloudy,2986.0
4,5,2026-01-01 02:28:00,Van B,8,Pune,18.946607,73.846363,18.679651,74.859443,27.7,Clear,1818.0


In [24]:
print("Rows and Columns:", df.shape)

print("\nColumn Names:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

Rows and Columns: (512, 12)

Column Names:
['Delivery ID', 'Starting Time', 'Van Model', 'Num Packages', 'Town', 'Start Lat', 'Start Lon', 'End Lat', 'End Lon', 'Temperature', 'Conditions', 'Delivery Time']

Data Types:
Delivery ID        int64
Starting Time        str
Van Model            str
Num Packages       int64
Town                 str
Start Lat        float64
Start Lon        float64
End Lat          float64
End Lon          float64
Temperature      float64
Conditions           str
Delivery Time    float64
dtype: object


In [25]:
df.describe(include="all")

,Delivery ID,Starting Time,Van Model,Num Packages,Town,Start Lat,Start Lon,End Lat,End Lon,Temperature,Conditions,Delivery Time
count,512.000000,512,512,512.000000,512,512.000000,512.000000,512.000000,512.000000,487.000000,497,503.000000
unique,NaN,500,3,NaN,9,NaN,NaN,NaN,NaN,NaN,3,NaN
top,NaN,2026-01-01 19:07:00,Van A,NaN,Mumbai,NaN,NaN,NaN,NaN,NaN,Cloudy,NaN
freq,NaN,2,173,NaN,129,NaN,NaN,NaN,NaN,NaN,177,NaN
mean,250.300781,NaN,NaN,15.035156,NaN,18.794440,74.315444,18.790973,74.306846,28.247228,NaN,3011.003976
std,144.697198,NaN,NaN,8.526621,NaN,0.235012,0.349527,0.232412,0.342019,4.967564,NaN,6327.032286
min,1.000000,NaN,NaN,-2.000000,NaN,18.400415,73.705167,18.400455,73.703249,13.700000,NaN,-100.000000
25%,124.750000,NaN,NaN,7.000000,NaN,18.584419,74.008099,18.599919,74.000715,24.700000,NaN,1657.500000
50%,250.500000,NaN,NaN,16.000000,NaN,18.782382,74.295214,18.778010,74.321797,28.100000,NaN,2393.000000
75%,374.250000,NaN,NaN,23.000000,NaN,19.000801,74.637282,18.997969,74.604486,31.800000,NaN,3059.000000


In [26]:
missing_percentage = (
    df.isna().mean() * 100
).round(2)

print("Missing Percentage by Column:")
print(missing_percentage[missing_percentage > 0])

Missing Percentage by Column:
Temperature      4.88
Conditions       2.93
Delivery Time    1.76
dtype: float64


In [27]:
duplicate_count = df.duplicated().sum()

print("Duplicate Rows:", duplicate_count)


Duplicate Rows: 12


In [28]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
)

print(df.columns.tolist())

['delivery_id', 'starting_time', 'van_model', 'num_packages', 'town', 'start_lat', 'start_lon', 'end_lat', 'end_lon', 'temperature', 'conditions', 'delivery_time']


In [29]:
df["starting_time"] = pd.to_datetime(
    df["starting_time"],
    errors="coerce"
)

print(df["starting_time"].head())
print("\nData Type:", df["starting_time"].dtype)

0   2026-01-01 00:00:00
1   2026-01-01 00:37:00
2   2026-01-01 01:14:00
3   2026-01-01 01:51:00
4   2026-01-01 02:28:00
Name: starting_time, dtype: datetime64[us]

Data Type: datetime64[us]


In [30]:
df["delivery_time"] = pd.to_numeric(
    df["delivery_time"],
    errors="coerce"
)

df["num_packages"] = pd.to_numeric(
    df["num_packages"],
    errors="coerce"
)

print(df[["delivery_time", "num_packages"]].dtypes)

delivery_time    float64
num_packages       int64
dtype: object


In [31]:
for col in ["town", "van_model", "conditions"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.title()
        )

print(df[["town", "van_model", "conditions"]].head())

          town van_model conditions
0       Mumbai     Van A      Clear
1  Ahilyanagar     Van C     Cloudy
2  Ahilyanagar     Van B       Rain
3       Mumbai     Van B     Cloudy
4         Pune     Van B      Clear


In [32]:
before_duplicates = len(df)

df = df.drop_duplicates()

after_duplicates = len(df)

print(
    f"Exact duplicates removed: "
    f"{before_duplicates - after_duplicates}"
)

print(
    f"Rows: {before_duplicates} -> {after_duplicates}"
)

Exact duplicates removed: 12
Rows: 512 -> 500


In [33]:
before_id_duplicates = len(df)

df = (
    df.sort_values("starting_time")
      .drop_duplicates(
          "delivery_id",
          keep="last"
      )
)

after_id_duplicates = len(df)

print(
    f"Duplicate delivery IDs removed: "
    f"{before_id_duplicates - after_id_duplicates}"
)

print(
    f"Rows: {before_id_duplicates} -> {after_id_duplicates}"
)

Duplicate delivery IDs removed: 0
Rows: 500 -> 500


In [34]:
invalid_delivery_time = df[
    df["delivery_time"] <= 0
]

print(
    "Invalid delivery-time records:",
    len(invalid_delivery_time)
)

Invalid delivery-time records: 4


In [35]:
invalid_packages = df[
    df["num_packages"] < 0
]

print(
    "Invalid package records:",
    len(invalid_packages)
)

Invalid package records: 3


In [36]:
before_invalid = len(df)

bad_records = (
    (df["delivery_time"] <= 0)
    | (df["num_packages"] < 0)
    | ~df["start_lat"].between(-90, 90)
    | ~df["end_lat"].between(-90, 90)
    | ~df["start_lon"].between(-180, 180)
    | ~df["end_lon"].between(-180, 180)
)

df = df[~bad_records.fillna(False)]

after_invalid = len(df)

print(
    f"Invalid records removed: "
    f"{before_invalid - after_invalid}"
)

print(
    f"Rows: {before_invalid} -> {after_invalid}"
)

Invalid records removed: 7
Rows: 500 -> 493


In [37]:
before_invalid = len(df)

bad_records = (
    (df["delivery_time"] <= 0)
    | (df["num_packages"] < 0)
    | ~df["start_lat"].between(-90, 90)
    | ~df["end_lat"].between(-90, 90)
    | ~df["start_lon"].between(-180, 180)
    | ~df["end_lon"].between(-180, 180)
)

df = df[~bad_records.fillna(False)]

after_invalid = len(df)

print(
    f"Invalid records removed: "
    f"{before_invalid - after_invalid}"
)

print(
    f"Rows: {before_invalid} -> {after_invalid}"
)

Invalid records removed: 0
Rows: 493 -> 493


In [38]:
missing = df.isnull().sum()

print(missing[missing > 0])

temperature      25
conditions       15
delivery_time     8
dtype: int64


In [40]:
before_target_missing = len(df)

df = df.dropna(
    subset=["delivery_time"]
)

after_target_missing = len(df)

print(
    f"Rows with missing delivery time removed: "
    f"{before_target_missing - after_target_missing}"
)

Rows with missing delivery time removed: 0
